# 02 — Descriptive statistics

Characterises the two labelled cohorts before any modelling: per-variable means
and standard deviations split by insulin-resistance status, the tests behind
those comparisons, the distribution of each variable, the correlation structure,
and the distribution of the HOMA-IR index itself.

Taiwan Biobank appears here only in the correlation panel. Its statistics table
and its boxplot need an `IR` label, which for that cohort is a *model
prediction* rather than a measurement, so both are produced in notebook 07.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
import polars as pl

from src.data.io import output_path, processed_path
from src.logging_utils import configure_logging
from src.viz.figures import (
    plot_correlation_matrix,
    plot_feature_boxplots,
    plot_homair_distributions,
)
from src.viz.palettes import COLOR_BLIND_DIVERGING, OKABE_ITO
from src.viz.tables import descriptive_statistics

configure_logging(ROOT / "logs")

nhanes = pl.read_parquet(processed_path("NHANES_data.parquet"))
knhanes = pl.read_parquet(processed_path("KNHANES_data.parquet"))
twb = pl.read_parquet(processed_path("TWB_clinical_data.parquet"))
combined = pl.concat([nhanes, knhanes])

print(f"NHANES {nhanes.shape}, KNHANES {knhanes.shape}, combined {combined.shape}, TWB {twb.shape}")

NHANES (11660, 21), KNHANES (15138, 21), combined (26798, 21), TWB (92734, 18)


## Cohort characteristics

Each variable is reported as `mean±SD` for the whole cohort and for each
insulin-resistance group, followed by four p-values: a Kolmogorov–Smirnov test
of each group against a normal distribution, Welch's t-test, and the
Mann–Whitney U test.

Two workbooks are written. `stats.xlsx` reports a p-value below 0.01 as
`"< 0.01"`, which is how such a table is usually printed. `stats_original.xlsx`
is identical except that every p-value is given as its own value, so a reader can
see how far below the threshold each result actually falls — the difference
between 0.009 and 1e-19 matters when judging which comparisons are strongest.

The equivalent table for Taiwan Biobank is in `stats_twb.xlsx`, written by
notebook 07: it is split by a predicted label, which needs a model that does not
exist yet at this point in the pipeline.

In [2]:
SHEETS = {"NHANES": nhanes, "KNHANES": knhanes, "COMBINE": combined}

tables = {name: descriptive_statistics(frame) for name, frame in SHEETS.items()}
tables_raw = {
    name: descriptive_statistics(frame, alpha=None) for name, frame in SHEETS.items()
}

for filename, sheets in [("stats.xlsx", tables), ("stats_original.xlsx", tables_raw)]:
    with pd.ExcelWriter(output_path(filename)) as writer:
        for name, table in sheets.items():
            table.to_excel(writer, sheet_name=name, index=False)

tables["COMBINE"]


2026-09-22 22:22:17 [INFO] src.viz.tables: Descriptive statistics table: 20 rows


2026-09-22 22:22:17 [INFO] src.viz.tables: Descriptive statistics table: 20 rows


2026-09-22 22:22:17 [INFO] src.viz.tables: Descriptive statistics table: 20 rows


2026-09-22 22:22:17 [INFO] src.viz.tables: Descriptive statistics table: 20 rows


2026-09-22 22:22:17 [INFO] src.viz.tables: Descriptive statistics table: 20 rows


2026-09-22 22:22:18 [INFO] src.viz.tables: Descriptive statistics table: 20 rows


,column,All,IR-,IR+,ks_p_value-,ks_p_value+,t_test_p_value,u_test_p_value
0,N,"26,798","17,389","9,409",None,None,None,None
1,Male,"12,320(46.0%)","7,568(43.5%)","4,752(50.5%)",None,None,None,None
2,Female,"14,478(54.0%)","9,821(56.5%)","4,657(49.5%)",None,None,None,None
3,AGE,48.8±17.8,48.6±17.6,49.4±18.0,< 0.01,< 0.01,< 0.01,< 0.01
4,BMI,25.7±5.3,23.9±3.8,29.2±5.9,< 0.01,< 0.01,< 0.01,< 0.01
5,BODY_WAISTLINE,89.1±14.3,84.1±11.2,98.5±14.5,< 0.01,< 0.01,< 0.01,< 0.01
6,BUN,13.9±5.0,14.0±4.9,13.9±5.2,< 0.01,< 0.01,0.14433164293085082,< 0.01
7,CREATININE,0.8±0.3,0.8±0.3,0.8±0.3,< 0.01,< 0.01,< 0.01,< 0.01
8,FASTING_GLUCOSE,98.6±16.9,94.2±10.2,106.7±22.8,< 0.01,< 0.01,< 0.01,< 0.01
9,FASTING_INSULIN,10.5±9.8,6.2±2.3,18.5±13.0,< 0.01,< 0.01,< 0.01,< 0.01


## Distributions, correlations and the HOMA-IR index

The boxplot grid uses the `RACE` column as a cohort label on the x axis rather
than the numeric ancestry code. Five right-skewed variables are shown on a
natural-log scale.

The correlation panel is the one place Taiwan Biobank appears in this notebook:
correlations need no label, so all four cohorts can be compared directly.

In [3]:
boxplot_data = pl.concat(
    [
        nhanes.with_columns(RACE=pl.lit("NHANES")),
        knhanes.with_columns(RACE=pl.lit("KNHANES")),
    ]
).select(
    pl.all().exclude(
        ["Release_No", "DIABETES", "SEX", "MET_ID", "FASTING_INSULIN", "HOMA-IR"]
    )
)
boxplot_data = boxplot_data.select(sorted(boxplot_data.columns))

plot_feature_boxplots(
    boxplot_data,
    "Boxplots of variables by IR and Race",
    output_path("boxplot_of_variable_by_IR_and_Race.png"),
);


2026-09-22 22:22:19 [INFO] src.viz.figures: Wrote output/boxplot_of_variable_by_IR_and_Race.png


In [4]:
FRAMES = {"NHANES": nhanes, "KNHANES": knhanes, "NHANES+KNHANES": combined, "TWB": twb}

plot_correlation_matrix(FRAMES, output_path("correlation_matrix.png"))
plot_correlation_matrix(
    FRAMES,
    output_path("correlation_matrix_color_blind.png"),
    cmap=COLOR_BLIND_DIVERGING,
);


2026-09-22 22:22:20 [INFO] src.viz.figures: Wrote output/correlation_matrix.png


2026-09-22 22:22:21 [INFO] src.viz.figures: Wrote output/correlation_matrix_color_blind.png


In [5]:
HOMAIR_FRAMES = {"NHANES": nhanes, "KNHANES": knhanes, "NHANES+KNHANES": combined}

plot_homair_distributions(HOMAIR_FRAMES, output_path("HOMA-IR.png"))
plot_homair_distributions(
    HOMAIR_FRAMES,
    output_path("HOMA-IR_color_blind.png"),
    cutoff_color=OKABE_ITO["vermillion"],
);


2026-09-22 22:22:22 [INFO] src.viz.figures: Wrote output/HOMA-IR.png


2026-09-22 22:22:23 [INFO] src.viz.figures: Wrote output/HOMA-IR_color_blind.png
